# 🧪 Running ORCA Jobs

---

## What is ORCA?

**ORCA** is a powerful **quantum chemistry software package** developed by the Max Planck Institute for Chemical Energy Conversion.  
It is widely used for performing **electronic structure calculations** and supports a wide range of computational chemistry methods such as:

- **Density Functional Theory (DFT)**
- **Hartree–Fock (HF)** and **post-HF** methods (MP2, CCSD, etc.)
- **Geometry optimization**, **vibrational frequency analysis**, and **excited-state simulations**

ORCA can efficiently utilize multiple CPU cores via **MPI (Message Passing Interface)** on our Anatra HPC cluster.  
This submission script automates the process of running ORCA 6.1.0 using **Slurm** with **OpenMPI**, providing researchers with a reliable and reproducible workflow for large-scale quantum chemistry simulations.

---


# Full ORCA Script

---

```bash
#!/bin/bash
set -euo pipefail

# Defaults
readonly DEFAULT_NPROCS=12
readonly DEFAULT_TIME="3-00:00:00"
readonly DEFAULT_PARTITION="chemistry"
readonly MAX_PROCS=96

nprocs=${DEFAULT_NPROCS}
time=${DEFAULT_TIME}
partition=${DEFAULT_PARTITION}

usage() {
    cat << EOF
ORCA 6.1.0 Job Submission Script (PMIx-Enabled)

USAGE: $(basename $0) <input_file> [input_file ...] [OPTIONS]

OPTIONS:
    -n <cores>      Number of CPU cores (default: ${DEFAULT_NPROCS}, max: ${MAX_PROCS})
    -t <time>       Time limit in dd-hh:mm:ss (default: ${DEFAULT_TIME})
    -p <partition>  SLURM partition (default: ${DEFAULT_PARTITION})
    -h              Display this help message
EOF
    exit 0
}

validate_input() {
    if ! [[ "$nprocs" =~ ^[0-9]+$ ]] || [ "$nprocs" -le 0 ]; then
        echo "ERROR: Number of cores must be a positive integer" >&2
        exit 1
    fi
    
    if [ "$nprocs" -gt "$MAX_PROCS" ]; then
        nprocs=$MAX_PROCS
    fi
    
    if ! [[ "$time" =~ ^[0-9]+-[0-9]{2}:[0-9]{2}:[0-9]{2}$ ]]; then
        echo "ERROR: Invalid time format. Use dd-hh:mm:ss" >&2
        exit 1
    fi
}

check_file_exists() {
    local file=$1
    if [ ! -f "$file" ] || [ ! -r "$file" ]; then
        echo "ERROR: File not found or not readable: $file" >&2
        return 1
    fi
    return 0
}

create_job_script() {
    local infile=$1
    local head="${infile%.*}"
    local job_script="${head}.slm"
    
    cat > "$job_script" <<'EOF'
    
#!/bin/bash
#SBATCH --job-name=HEAD_PLACEHOLDER
#SBATCH --partition=PARTITION_PLACEHOLDER
#SBATCH --output=HEAD_PLACEHOLDER.%j.out
#SBATCH --error=HEAD_PLACEHOLDER.%j.err
#SBATCH --time=TIME_PLACEHOLDER
#SBATCH --nodes=1
#SBATCH --ntasks=NPROCS_PLACEHOLDER
#SBATCH --cpus-per-task=1

set -euo pipefail

# Load modules
module purge
module load GCC/15.1.0 openmpi/4.1.6
module load orca/6.1.0

# PMIx configuration
export SLURM_MPI_TYPE=pmix
export OMPI_MCA_pmix_base_verbose=0
export ORCA_MPI_RSH="srun"

# Setup paths
logfile="$(pwd)/HEAD_PLACEHOLDER.out"
workdir="$(pwd)"
tempdir="/tmp/jobs/${SLURM_JOB_ID}/HEAD_PLACEHOLDER.files"

mkdir -p "$tempdir" || { echo "ERROR: Cannot create temp directory"; exit 1; }
cp "$workdir/HEAD_PLACEHOLDER"* "$tempdir" || { echo "ERROR: Cannot copy input files"; exit 1; }
cd "$tempdir" || { echo "ERROR: Cannot change to temp directory"; exit 1; }

# Run ORCA
$(which orca) INFILE_PLACEHOLDER 1>>"$logfile" 2>&1
orca_exit=$?

# Cleanup and copy results
find "$tempdir" -maxdepth 1 -type f -name "*.tmp*" -delete
cp -Rf "$tempdir"/* "$workdir/" || { echo "ERROR: Cannot copy results back"; exit 1; }

exit $orca_exit
EOF

    # Replace placeholders
    sed -i "s|HEAD_PLACEHOLDER|${head}|g" "$job_script"
    sed -i "s|PARTITION_PLACEHOLDER|${partition}|g" "$job_script"
    sed -i "s|TIME_PLACEHOLDER|${time}|g" "$job_script"
    sed -i "s|NPROCS_PLACEHOLDER|${nprocs}|g" "$job_script"
    sed -i "s|INFILE_PLACEHOLDER|${infile}|g" "$job_script"

    chmod +x "$job_script"
    echo "$job_script"
}

# Parse arguments
if [ $# -eq 0 ]; then
    usage
fi

files=()
while [[ $# -gt 0 && "$1" == *.inp ]]; do
    files+=("$1")
    shift
done

while getopts "n:t:p:h" opt; do
    case "$opt" in
        n) nprocs=$OPTARG ;;
        t) time=$OPTARG ;;
        p) partition=$OPTARG ;;
        h) usage ;;
        *) usage ;;
    esac
done

validate_input

if [ ${#files[@]} -eq 0 ]; then
    echo "ERROR: No input files provided" >&2
    usage
fi

# Process files
submitted_jobs=0
failed_jobs=0

for infile in "${files[@]}"; do
    if [[ "$infile" == *.inp ]]; then
        if ! check_file_exists "$infile"; then
            ((failed_jobs++))
            continue
        fi
        
        job_script=$(create_job_script "$infile")
        
        if job_id=$(sbatch "$job_script" 2>&1); then
            echo "Submitted: $infile -> $job_id"
            ((submitted_jobs++))
        else
            echo "Failed: $infile" >&2
            ((failed_jobs++))
        fi
    fi
done

echo "Summary: $submitted_jobs submitted, $failed_jobs failed"
exit 0


## ORCA Job Submission Guide for Anatra Cluster

### How to Run the Script

Follow these steps to run your ORCA jobs using this submission script on the HPC cluster.

---

### 🧩 Step 1 — Save the Script

Save the code above as:

```bash
submit_orca.sh
```

Then make it executable:

```bash
chmod +x submit_orca.sh
```

---

### 🧪 Step 2 — Prepare Your Input File

Create your ORCA input file, for example:

```bash
molecule.inp
```

---

### 🚀 Step 3 — Submit a Job

Submit a single job:

```bash
./submit_orca.sh molecule.inp -n 24 -t 2-00:00:00 -p chemistry
```

Or multiple jobs:

```bash
./submit_orca.sh job1.inp job2.inp -n 16 -t 1-12:00:00
```

---

### 📊 Step 4 — Monitor Job Progress

Check the status of your jobs:

```bash
squeue -u $USER
```

To view output logs:

```bash
cat molecule.%j.out
```

---

### 🧭 Step 5 — Cancel a Job (if needed)

Cancel a specific job:

```bash
scancel <job_id>
```

---

### 📂 Step 6 — Check Output Files

After the job finishes, the following files will appear in your working directory:

```
molecule.out
molecule.%j.out
molecule.%j.err
molecule.gbw
```

These files contain ORCA's output data and calculation logs.

---

## 📝 Notes

- Replace `<job_id>` with your actual SLURM job ID when canceling jobs
- The `%j` in filenames will be automatically replaced with your job ID number
- Adjust the number of cores (`-n`) and time limit (`-t`) based on your calculation requirements
- Make sure to specify the correct partition (`-p`) for your institution's HPC setup